Imports

In [3]:
import cv2
import os
import random
import numpy as np
import tensorflow as tf
from keras._tf_keras.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Lambda
from keras._tf_keras.keras.models import Model
import keras._tf_keras.keras.backend as K
from keras._tf_keras.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

Constants

In [4]:
# Image dimensions
IMG_SIZE = (128, 256, 1)  # (Height, Width, Channels)

Definitions

In [5]:
# Define the CNN feature extractor
def build_base_cnn(input_shape=IMG_SIZE):
    inputs = Input(shape=input_shape)
    x = Conv2D(64, (3,3), activation='relu', padding='same')(inputs)
    x = MaxPooling2D(pool_size=(2,2))(x)
    x = Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = MaxPooling2D(pool_size=(2,2))(x)
    x = Flatten()(x)
    x = Dense(256, activation='relu')(x)
    return Model(inputs, x)

Siamese Network Implementation

In [6]:
# Create the base network
base_network = build_base_cnn()

# Inputs for the two images
input_1 = Input(shape=IMG_SIZE)
input_2 = Input(shape=IMG_SIZE)

# Feature extraction
feature_1 = base_network(input_1)
feature_2 = base_network(input_2)

# Calculate L1 Distance between feature vectors
def euclidean_distance(vectors):
    x, y = vectors
    return K.sqrt(K.sum(K.square(x - y), axis=1, keepdims=True))

distance = Lambda(euclidean_distance)([feature_1, feature_2])

# Final output layer
output = Dense(1, activation='sigmoid')(distance)

# Siamese Network Model
siamese_model = Model(inputs=[input_1, input_2], outputs=output)
siamese_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

siamese_model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 128, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 128, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional          │ (None, 256)       │ 67,183,616 │ input_layer_1[0]… │
│ (Functional)        │                   │            │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 1)         │          0 │ functional[0][0], │
│                     │                   │            │ functional[1][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │          2 │ lambda[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 67,183,618 (256.29 MB)

 Trainable params: 67,183,618 (256.29 MB)

 Non-trainable params: 0 (0.00 B)

Generating Pairs

In [16]:
# Path to dataset
DATASET_PATH = "/Users/pasan_diksura/Documents/SoftwareDevelopment/Projects/Signature-Forgery-Detection-System/Signature-Forgery-Detection-System/Dataset/zProcessed"

# Load images into a dictionary by person ID
def load_images():
    images = {}

    for filename in os.listdir(DATASET_PATH):
        if filename.endswith(".jpg") or filename.lower().endswith(".jpeg"):
            parts = filename.split("_")
            label, person_id, img_id = parts[0], parts[2], parts[3]
            full_path = os.path.join(DATASET_PATH, filename)
            if person_id not in images:
                images[person_id] = {"T": [], "F": []}
            images[person_id][label].append(full_path)
    return images

images_dict = load_images()

# Create positive and negative pairs
def create_pairs(images_dict, num_pairs=1000, img_size=(128, 256)):
    positive_pairs, negative_pairs, labels = [], [], []

    person_ids = list(images_dict.keys())

    for person_id in person_ids:
        genuine_signatures = images_dict[person_id]["T"]
        forged_signatures = images_dict[person_id]["F"]

        # Create Positive Pairs (Genuine-Genuine)
        if len(genuine_signatures) > 1:
            for _ in range(num_pairs // 2):
                img1, img2 = random.sample(genuine_signatures, 2)
                positive_pairs.append((img1, img2))
                labels.append(1)

        if not genuine_signatures:  # Check if genuine signatures list is empty
            continue

        # Create Negative Pairs (Genuine-Forged from different persons)
        for _ in range(num_pairs // 2):
            person_2 = random.choice([p for p in person_ids if p != person_id])
            img1 = random.choice(genuine_signatures)
            img2 = random.choice(images_dict[person_2]["F"])
            negative_pairs.append((img1, img2))
            labels.append(0)

    all_pairs = positive_pairs + negative_pairs
    return np.array(all_pairs), np.array(labels)

pairs, labels = create_pairs(images_dict)

# Split into train/test
train_pairs, test_pairs, train_labels, test_labels = train_test_split(
    pairs, labels, test_size=0.2, random_state=42
)

# Convert images to numpy arrays
def process_image_pairs(pairs, img_size=(128, 256)):
    X1, X2 = [], []
    for img1, img2 in pairs:
        img1 = img_to_array(load_img(img1, target_size=img_size)) / 255.0
        img2 = img_to_array(load_img(img2, target_size=img_size)) / 255.0
        X1.append(img1)
        X2.append(img2)
    return np.array(X1), np.array(X2)

X_train_1, X_train_2 = process_image_pairs(train_pairs)
X_test_1, X_test_2 = process_image_pairs(test_pairs)

y_train, y_test = train_labels, test_labels

# Save preprocessed data
np.save("X_train_1.npy", X_train_1)
np.save("X_train_2.npy", X_train_2)
np.save("y_train.npy", y_train)
np.save("X_test_1.npy", X_test_1)
np.save("X_test_2.npy", X_test_2)
np.save("y_test.npy", y_test)

print("Data Preprocessing Complete! Ready for Model Training.")

Data Preprocessing Complete! Ready for Model Training.


Building the Model

In [17]:
# Load preprocessed data
X_train_1 = np.load("X_train_1.npy")
X_train_2 = np.load("X_train_2.npy")
y_train = np.load("y_train.npy")

X_test_1 = np.load("X_test_1.npy")
X_test_2 = np.load("X_test_2.npy")
y_test = np.load("y_test.npy")

# Image shape
IMG_SHAPE = (128, 256, 3)

# Define the CNN model for feature extraction
def create_base_network(input_shape):
    input_layer = Input(shape=input_shape)
    
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(input_layer)
    x = MaxPooling2D()(x)
    
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D()(x)
    
    x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D()(x)

    x = Flatten()(x)
    x = Dense(512, activation='relu')(x)

    return Model(input_layer, x)

# Define the Siamese Network
base_network = create_base_network(IMG_SHAPE)

input_a = Input(shape=IMG_SHAPE)
input_b = Input(shape=IMG_SHAPE)

feat_a = base_network(input_a)
feat_b = base_network(input_b)

# Compute L1 distance
l1_distance = Lambda(lambda tensors: tf.abs(tensors[0] - tensors[1]))([feat_a, feat_b])

# Classification layer
output_layer = Dense(1, activation='sigmoid')(l1_distance)

# Compile model
siamese_model = Model(inputs=[input_a, input_b], outputs=output_layer)
siamese_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
siamese_model.fit(
    [X_train_1, X_train_2], y_train,
    validation_data=([X_test_1, X_test_2], y_test),
    batch_size=32,
    epochs=20
)

# Save model
siamese_model.save("siamese_signature_model.keras")

print(" Model Training Complete! Ready for Evaluation.")

Epoch 1/20
1675/1675 ━━━━━━━━━━━━━━━━━━━━ 1889s 1s/step - accuracy: 0.7742 - loss: 0.4548 - val_accuracy: 0.9704 - val_loss: 0.0850
Epoch 2/20
 130/1675 ━━━━━━━━━━━━━━━━━━━━ 26:11 1s/step - accuracy: 0.9882 - loss: 0.0399

KeyboardInterrupt: 

Model Evaluation

In [ ]:
# Load model
siamese_model = tf.keras.models.load_model("siamese_signature_model.keras")

# Get predictions (probabilities)
y_pred_prob = siamese_model.predict([X_test_1, X_test_2])

# Convert probabilities to binary labels (Threshold = 0.5)
y_pred = (y_pred_prob > 0.5).astype(int)

# Evaluate performance
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

# Detailed classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))


Making Predictions on New Images

In [ ]:
# Function to preprocess the input images
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    image = cv2.resize(image, (128, 256))  # Resize to match training shape
    image = image / 255.0  # Normalize
    return np.expand_dims(image, axis=0)  # Add batch dimension

# Load new signature images
img1 = preprocess_image("test_signature1.jpg")
img2 = preprocess_image("test_signature2.jpg")

# Make prediction
prediction = siamese_model.predict([img1, img2])

# Print result
if prediction > 0.5:
    print("Signatures Match (Genuine)")
else:
    print("Signatures Do Not Match (Forgery)")
